# Q2A | Summarize the Data

> *Summarize the data. How much data is present? What attributes/features are continuous valued? Which attributes are categorical?*

This notebook answers the question in three parts:

1. **How much data is present?** Row and column counts, duplicates, and missing values.
2. **Which attributes are continuous?** Data types, the number of distinct values, the range of each numeric attribute, and the date column.
3. **Which attributes are categorical?** The values each categorical attribute takes, and the one column that is neither continuous nor categorical.

The **Answer Summary** below pulls the three parts together.

---

#### **IMPORTANT**

This dataset is a single file with no `train` / `test` split. The split is made in a later notebook, so this summary describes the whole dataset. It only counts and describes the data. No modeling decisions are made here, so looking at rows that will later become test data does not introduce `data snooping` bias.

# Answer Summary

- **How much data is present?** 731 rows and 16 columns (1 target, `Rentals`, and 15 other columns) in a single file with no train/test split. Each row is one day, from 2011-01-01 to 2012-12-31. There are no duplicate rows and no missing values.
- **Which attributes are continuous?** 8. The four measurements `Temp`, `FeelsLike`, `Humidity`, and `Wind` hold decimals between 0 and 1. The three rental counts `Casual`, `Registered`, and `Rentals` are whole-number amounts with 606 to 696 distinct values. `Date` is a point in time, with 731 distinct values spaced exactly 1 day apart.
- **Which attributes are categorical?** 7: `Season`, `Year`, `Month`, `IsHoliday`, `Weekday`, `IsWorkday`, and `Weather`. Each takes 2 to 12 distinct values that label a group of days. The remaining column, `ID`, is neither: it numbers the rows and has a different value in every row.

---

# Imports

In [1]:
import pandas as pd

from Q2.consts import *

from utils import read_csv_to_df

# Fetch Data

In [2]:
df = read_csv_to_df(path=DATA_PATH, parse_dates=["dteday"])
df = df.rename(columns=SHORT_NAMES)

# 1. How Much Data is Present?

In [3]:
summary = pd.DataFrame(
	{
		"row count": [ df.shape[0] ],
		"columns count": [ df.shape[1] ],
		"missing values": [ df.isnull().sum().sum() ],
		"rows w/ missing values": [ df.isnull().any(axis=1).sum() ],
		"% rows w/ missing values": [
			round(df.isnull().any(axis=1).sum() / df.shape[0] * 100, 2),
		]
	},
	index=["Full Dataset"],
)
s = summary.T
display(s)

df.info()

,Full Dataset
row count,731.0
columns count,16.0
missing values,0.0
rows w/ missing values,0.0
% rows w/ missing values,0.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   ID          731 non-null    int64         
 1   Date        731 non-null    datetime64[ns]
 2   Season      731 non-null    int64         
 3   Year        731 non-null    int64         
 4   Month       731 non-null    int64         
 5   IsHoliday   731 non-null    int64         
 6   Weekday     731 non-null    int64         
 7   IsWorkday   731 non-null    int64         
 8   Weather     731 non-null    int64         
 9   Temp        731 non-null    float64       
 10  FeelsLike   731 non-null    float64       
 11  Humidity    731 non-null    float64       
 12  Wind        731 non-null    float64       
 13  Casual      731 non-null    int64         
 14  Registered  731 non-null    int64         
 15  Rentals     731 non-null    int64         
dtypes: datetime64[ns](1), floa

- There are 731 rows and 16 columns (1 target, `Rentals`, and 15 other columns).
- There are 0 missing values.
- 15 columns hold numbers (`int64` or `float64`) and 1 column, `Date`, holds dates.

## Duplicate Rows

In [4]:
df.duplicated().sum()

0

- There are no duplicated rows.
- Every row is a distinct observation.

# 2. Which Attributes are Continuous?

An attribute is continuous if it measures a quantity that can take any value within a range. That depends on what the values mean, not on how they are stored, so two checks are used as evidence:

1. **Are the values numeric?** A continuous attribute must hold numbers, but the stored data type alone is not proof either way. Numbers can be stored as text and categories can be stored as numbers.
2. **Do the values behave like measurements?** Many distinct values, fractional values, and a spread across a range point to a measured quantity. A handful of repeated values points to codes or categories.

In [5]:
info_df = pd.DataFrame({
	"dtype": df.dtypes,
	"non-null values": df.count(),
	"unique values": df.nunique(),
	"most frequent value count": df.apply(lambda col: col.value_counts().iloc[0]),
}).sort_values(by=["unique values", "most frequent value count"], ascending=False)

info_df

,dtype,non-null values,unique values,most frequent value count
ID,int64,731,731,1
Date,datetime64[ns],731,731,1
Rentals,int64,731,696,2
FeelsLike,float64,731,690,4
Registered,int64,731,679,3
Wind,float64,731,650,3
Casual,int64,731,606,4
Humidity,float64,731,595,4
Temp,float64,731,499,5
Month,int64,731,12,62


- 15 of the 16 columns are numeric (11 `int64` and 4 `float64`). The other column, `Date`, holds dates. It is checked on its own in the Date section below.
- The number of distinct values splits the numeric columns into two clear groups:
  - 7 columns have 12 or fewer distinct values: `Season` (4), `Year` (2), `Month` (12), `IsHoliday` (2), `Weekday` (7), `IsWorkday` (2), and `Weather` (3).
  - 8 columns have 499 or more: `ID` (731), `Temp` (499), `FeelsLike` (690), `Humidity` (595), `Wind` (650), `Casual` (606), `Registered` (679), and `Rentals` (696). No single value dominates this group. The most common value in any of those columns repeats at most 5 times in 731 rows.
- `ID` and `Date` each have 731 distinct values in 731 rows: every row has its own value.

## Value Ranges

In [6]:
numeric_df = df.select_dtypes(include="number")

value_ranges = pd.DataFrame({
	"min": numeric_df.min().round(3),
	"max": numeric_df.max().round(3),
	"mean": numeric_df.mean().round(3),
	"std": numeric_df.std().round(3),
	"whole numbers only": numeric_df.apply(lambda col: (col.dropna() % 1 == 0).all()),
}, index=numeric_df.columns)

value_ranges

,min,max,mean,std,whole numbers only
ID,1.000,731.000,366.000,211.166,True
Season,1.000,4.000,2.497,1.111,True
Year,0.000,1.000,0.501,0.500,True
Month,1.000,12.000,6.520,3.452,True
IsHoliday,0.000,1.000,0.029,0.167,True
Weekday,0.000,6.000,2.997,2.005,True
IsWorkday,0.000,1.000,0.684,0.465,True
Weather,1.000,3.000,1.395,0.545,True
Temp,0.059,0.862,0.495,0.183,False
FeelsLike,0.079,0.841,0.474,0.163,False


Only the 15 numeric columns are shown. `Date` is covered in the next section.

- 4 columns hold decimals: `Temp`, `FeelsLike`, `Humidity`, and `Wind`. The other 11 hold whole numbers only.
- Measurements: all four decimal columns sit between 0 and 1. `Temp` runs from 0.059 to 0.862, `FeelsLike` from 0.079 to 0.841, `Humidity` from 0.0 to 0.972, and `Wind` from 0.022 to 0.507.
- Counts: `Casual` (2 to 3,410), `Registered` (20 to 6,946), and the target `Rentals` (22 to 8,714) are whole numbers spread over a wide range.
- `ID` runs from 1 to 731, the same as the number of rows. It has many distinct values only because it numbers the rows, so it is not a measurement and is not counted as continuous.
- The other 7 whole-number columns (`Season`, `Year`, `Month`, `IsHoliday`, `Weekday`, `IsWorkday`, and `Weather`) each stay in a short fixed range (ex: 1 to 4 for `Season`), so they are not continuous. Section 3 covers them.

7 of the 15 numeric columns are continuous: the 4 measurements (`Temp`, `FeelsLike`, `Humidity`, and `Wind`) and the 3 rental counts (`Casual`, `Registered`, and `Rentals`). The counts are whole numbers, so they cannot take every value in their range. They are still amounts, not labels, with 606 to 696 distinct values, so they are grouped with the continuous attributes and not with the categorical ones.

## Date

In [7]:
days_since_first = (df["Date"] - df["Date"].min()).dt.days

date_summary = pd.DataFrame(
	{
		"first date": [ df["Date"].min().date() ],
		"last date": [ df["Date"].max().date() ],
		"distinct dates": [ df["Date"].nunique() ],
		"days from first to last date": [ days_since_first.max() ],
		"rows 1 day after the previous row": [ (df["Date"].diff().dt.days == 1).sum() ],
		"rows where ID = days since first date + 1": [ (df["ID"] == days_since_first + 1).sum() ],
	},
	index=["Date"],
)

date_summary.T

,Date
first date,2011-01-01
last date,2012-12-31
distinct dates,731
days from first to last date,730
rows 1 day after the previous row,730
rows where ID = days since first date + 1,731


- `Date` runs from 2011-01-01 to 2012-12-31, a span of 730 days, with 731 distinct dates in 731 rows.
- The rows are evenly spaced. Each of the 730 rows after the first is exactly 1 day after the row before it, so each row is one day and no day is skipped.
- A date is a point in time. Time is a quantity that can take any value within a range, and the gap between two dates is an amount (a number of days). `Date` is stored as a date, but it becomes a number once it is counted from a starting point (ex: days since the first date).
- Like the rental counts, `Date` is recorded in whole steps (1 day), so it does not take every value in its range.
- `ID` equals the days since the first date plus 1 in all 731 rows. It matches `Date` only because the rows are in date order with one row per day. It is still a row number, a label for the row, so it is not counted as continuous.

**8 attributes are continuous: the 4 measurements (`Temp`, `FeelsLike`, `Humidity`, and `Wind`), the 3 rental counts (`Casual`, `Registered`, and `Rentals`), and `Date`.**

# 3. Which Attributes are Categorical?

An attribute is categorical if its values name a group that a row belongs to. The value is a label, not an amount. That depends on what the values mean, not on how they are stored, so two checks are used as evidence:

1. **Do the values come from a small fixed set?** A categorical attribute has a few distinct values, and each one repeats across many rows. Section 2 showed that 7 columns have 12 or fewer distinct values.
2. **Do the values stand for groups and not amounts?** A category can be stored as a number (ex: 1, 2, 3, 4 for `Season`). The number is then a name for the group, so a value of 4 is not twice as much as a value of 2.

In [8]:
categorical_cols = ["Season", "Year", "Month", "IsHoliday", "Weekday", "IsWorkday", "Weather"]

category_levels = pd.DataFrame({
	"distinct values": df[categorical_cols].nunique(),
	"values": [", ".join(map(str, sorted(df[col].unique()))) for col in categorical_cols],
	"fewest rows per value": [df[col].value_counts().min() for col in categorical_cols],
	"most rows per value": [df[col].value_counts().max() for col in categorical_cols],
})

category_levels

,distinct values,values,fewest rows per value,most rows per value
Season,4,"1, 2, 3, 4",178,188
Year,2,"0, 1",365,366
Month,12,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12",57,62
IsHoliday,2,"0, 1",21,710
Weekday,7,"0, 1, 2, 3, 4, 5, 6",104,105
IsWorkday,2,"0, 1",231,500
Weather,3,"1, 2, 3",21,463


- All 7 columns pass the first check. Each has 2 to 12 distinct values, and every value repeats across many rows. The rarest value in any column still appears in 21 of the 731 rows (`IsHoliday` and `Weather`).
- All 7 pass the second check. The values are short runs of whole numbers (ex: 1 to 4 for `Season`, 0 to 6 for `Weekday`), and the column names describe groups of days (a season, a year, a month, a day of the week, a weather type, or a yes/no flag). The numbers are labels, not amounts.
- Whole numbers alone do not make a column categorical. `Casual`, `Registered`, and `Rentals` are also whole numbers, but they are amounts with 606 to 696 distinct values, so they stay continuous.
- `Date` is not categorical. None of its 731 values repeats, so it does not group rows. Section 2 counts it as continuous.

**7 attributes are categorical: `Season`, `Year`, `Month`, `IsHoliday`, `Weekday`, `IsWorkday`, and `Weather`.**

**1 column is neither continuous nor categorical:**

- `ID` has 731 distinct values in 731 rows and runs from 1 to 731. It numbers the rows, so it is not a measured amount, and no two rows share a value, so it does not group rows.

With the 8 continuous attributes from section 2, that accounts for all 16 columns (8 continuous, 7 categorical, 1 neither).